In [1]:
import pandas as pd

sample_df = pd.read_csv("../data/processed/sample_dataset.csv")

sample_df.head()

,heading,news_body,category
0,नेप्से र कारोबार रकम दुवै घट्यो,काठमाडौं — आइतबार र सोमबार उच्च अंकले बढेको ने...,business
1,लुम्बिनी घुम्न निम्तो,काठमाडौँ — यो वर्ष प्रदेश ५ ले लुम्बिनीमा सकेक...,business
2,कृषि अनुदान ‘दुरुपयोग’,काठमाडौँ — रुकुमका राजकुमार बुढामगर पेसाले किस...,business
3,नेपाल-चीन भन्सार २६ औं बैठक,धुलिखेल — नेपाल-चीनका भन्सार अधिकारीहरूको वषिर...,business
4,चियाका समस्या सम्बोधन गर्न कार्यदल,काठमाडौ — मुलुककै सबैभन्दा पुरानो निर्यातयोग्य...,business


In [2]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove unusual Unicode separators
    text = re.sub(r"[\u2028\u2029]", " ", text)

    # Replace newlines and tabs
    text = re.sub(r"[\r\n\t]", " ", text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [3]:
sample_df["clean_text"] = sample_df["news_body"].apply(clean_text)

In [4]:
sample_df["heading"] = sample_df["heading"].fillna("")

In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

sample_df["label"] = label_encoder.fit_transform(
    sample_df["category"]
)

sample_df.head()

,heading,news_body,category,clean_text,label
0,नेप्से र कारोबार रकम दुवै घट्यो,काठमाडौं — आइतबार र सोमबार उच्च अंकले बढेको ने...,business,काठमाडौं — आइतबार र सोमबार उच्च अंकले बढेको ने...,0
1,लुम्बिनी घुम्न निम्तो,काठमाडौँ — यो वर्ष प्रदेश ५ ले लुम्बिनीमा सकेक...,business,काठमाडौँ — यो वर्ष प्रदेश ५ ले लुम्बिनीमा सकेक...,0
2,कृषि अनुदान ‘दुरुपयोग’,काठमाडौँ — रुकुमका राजकुमार बुढामगर पेसाले किस...,business,काठमाडौँ — रुकुमका राजकुमार बुढामगर पेसाले किस...,0
3,नेपाल-चीन भन्सार २६ औं बैठक,धुलिखेल — नेपाल-चीनका भन्सार अधिकारीहरूको वषिर...,business,धुलिखेल — नेपाल-चीनका भन्सार अधिकारीहरूको वषिर...,0
4,चियाका समस्या सम्बोधन गर्न कार्यदल,काठमाडौ — मुलुककै सबैभन्दा पुरानो निर्यातयोग्य...,business,काठमाडौ — मुलुककै सबैभन्दा पुरानो निर्यातयोग्य...,0


In [6]:
mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

mapping

{'business': np.int64(0),
 'crime': np.int64(1),
 'economy': np.int64(2),
 'education': np.int64(3),
 'entertainment': np.int64(4),
 'global': np.int64(5),
 'health': np.int64(6),
 'national': np.int64(7),
 'politics': np.int64(8),
 'science and technology': np.int64(9),
 'society': np.int64(10),
 'sports': np.int64(11)}

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    sample_df["clean_text"],
    sample_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=sample_df["label"]
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

In [9]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(label_encoder, "../models/label_encoder.pkl")

['../models/label_encoder.pkl']